# MotionBricks — Kinematic Trajectory Generation

Generates qpos sequences for each motion style and saves them to Google Drive.

**Runtime:** GPU (T4 or better). Runtime → Change runtime type → T4 GPU.

**Output:** `MyDrive/cs348k/data/motionbricks/` — one `.npy` per clip + `motion_labels.npy`.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
OUTPUT_DIR = '/content/drive/MyDrive/cs348k/data/motionbricks'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output dir: {OUTPUT_DIR}')

## 2. Clone repo + pull LFS checkpoints (~2.3 GB, ~5–10 min)

In [ ]:
%%bash
set -e
apt-get install -q git-lfs
git lfs install

if [ ! -d /content/GR00T-WholeBodyControl ]; then
  GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/NVlabs/GR00T-WholeBodyControl.git /content/GR00T-WholeBodyControl
else
  echo 'Repo already cloned'
fi

cd /content/GR00T-WholeBodyControl/motionbricks

BASE="https://media.githubusercontent.com/media/NVlabs/GR00T-WholeBodyControl/main/motionbricks"

echo "Downloading checkpoints directly (bypasses git-lfs which fails silently in Colab)..."
curl -L --progress-bar "$BASE/out/G1-clip.ckpt" -o out/G1-clip.ckpt
curl -L --progress-bar "$BASE/out/motionbricks_vqvae/version_1/checkpoints/model-step=2000000.ckpt" \
     -o out/motionbricks_vqvae/version_1/checkpoints/model-step=2000000.ckpt
curl -L --progress-bar "$BASE/out/motionbricks_root/version_1/checkpoints/model-step=2000000.ckpt" \
     -o out/motionbricks_root/version_1/checkpoints/model-step=2000000.ckpt
curl -L --progress-bar "$BASE/out/motionbricks_pose/version_1/checkpoints/model-step=2000000.ckpt" \
     -o out/motionbricks_pose/version_1/checkpoints/model-step=2000000.ckpt

echo "=== Checkpoint sizes ==="
ls -lh out/G1-clip.ckpt
ls -lh out/motionbricks_vqvae/version_1/checkpoints/*.ckpt
ls -lh out/motionbricks_root/version_1/checkpoints/*.ckpt
ls -lh out/motionbricks_pose/version_1/checkpoints/*.ckpt

## 3. Install MotionBricks

In [ ]:
%%bash
cd /content/GR00T-WholeBodyControl/motionbricks
pip install -q -e .
echo 'Install done'

## 4. Load model (once — ~1–2 min)

In [ ]:
import sys, os
import numpy as np
import torch

from unittest.mock import MagicMock
for mod in ['pynput', 'pynput.keyboard', 'pynput.mouse', 'pynput._util',
            'pynput._util.xorg', 'pynput._util.darwin', 'pynput._util.win32']:
    sys.modules[mod] = MagicMock()

if not getattr(torch, '_load_patched', False):
    _orig_torch_load = torch.load
    def _patched_torch_load(f, *args, **kwargs):
        kwargs.setdefault('weights_only', False)
        return _orig_torch_load(f, *args, **kwargs)
    torch.load = _patched_torch_load
    torch._load_patched = True

os.chdir('/content/GR00T-WholeBodyControl/motionbricks')
sys.path.insert(0, '.')

from motionbricks.motion_backbone.demo.utils import navigation_demo
import argparse

args = argparse.Namespace(
    controller='random', has_viewer=0, use_qpos=1, clips='G1',
    max_steps=10000, random_seed=42, num_runs=1, generate_dt=2.0,
    lookat_movement_direction=0, pre_filter_qpos=1,
    source_root_realignment=1, target_root_realignment=1,
    force_canonicalization=1, skip_ending_target_cond=0,
    random_speed_scale=0, speed_scale=[1.0, 1.0], reprocess_clips=0,
    return_model_configs=True, return_dataloader=True,
    recording_dir=None, EXP='default',
)

print('Loading MotionBricks... (this takes ~1-2 min)')
demo_agent = navigation_demo(args)
print('Model ready.')

## 5. Generate motion clips

In [ ]:
import mujoco

# Actual modes from clip_holder_G1 — use these exact strings
MOTION_CONFIGS = [
    # (allowed_mode,       n_frames, category)
    ('idle',               150,      'static'),
    ('walk',               200,      'locomotion'),
    ('slow_walk',          200,      'locomotion'),
    ('stealth_walk',       200,      'locomotion'),
    ('injured_walk',       200,      'locomotion'),
    ('walk_zombie',        200,      'locomotion'),
    ('walk_boxing',        180,      'expressive'),
    ('walk_happy_dance',   180,      'expressive'),
    ('walk_gun',           180,      'expressive'),
    ('walk_scared',        180,      'expressive'),
    ('hand_crawling',      150,      'whole_body'),
    ('elbow_crawling',     150,      'whole_body'),
    ('walk_stealth',       180,      'locomotion'),
]

N_SEEDS = 3  # clips per mode


def generate_clip(demo_agent, mode, n_frames, seed):
    """Returns (n_frames, 36) float32 qpos array."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    demo_agent.full_agent.reset()

    qpos_list = []
    for step in range(n_frames):
        qpos = demo_agent.full_agent.get_next_frame()
        qpos_list.append(qpos.copy())
        demo_agent.mj_data.qpos[:] = qpos

        control_signals = demo_agent.controller.generate_control_signals(
            None, demo_agent.mj_model, demo_agent.mj_data,
            visualize=False,
            control_info={'force_idle': False, 'allowed_mode': mode}
        )
        if args.use_qpos:
            control_signals['context_mujoco_qpos'] = demo_agent.full_agent.get_context_mujoco_qpos()

        with torch.no_grad():
            demo_agent.full_agent.generate_new_frames(
                control_signals,
                demo_agent.controller.get_controller_dt() * args.generate_dt
            )
        mujoco.mj_forward(demo_agent.mj_model, demo_agent.mj_data)

    return np.stack(qpos_list).astype(np.float32)


labels = {}
for mode, n_frames, category in MOTION_CONFIGS:
    for seed in range(N_SEEDS):
        clip_name = f'{mode}_seed{seed}'
        print(f'  {clip_name} ({n_frames} frames)...', end=' ', flush=True)
        try:
            qpos = generate_clip(demo_agent, mode, n_frames, seed=seed * 1000)
            out_path = os.path.join(OUTPUT_DIR, f'{clip_name}.npy')
            np.save(out_path, qpos)
            labels[clip_name] = category
            print(f'OK shape={qpos.shape}')
        except Exception as e:
            print(f'FAILED: {e}')

np.save(os.path.join(OUTPUT_DIR, 'motion_labels.npy'), labels)
print(f'\nDone. {len(labels)} clips saved to {OUTPUT_DIR}')

## 6. Verify outputs

In [ ]:
files = sorted(f for f in os.listdir(OUTPUT_DIR) if f.endswith('.npy') and f != 'motion_labels.npy')
print(f'{len(files)} clips:')
for f in files:
    arr = np.load(os.path.join(OUTPUT_DIR, f))
    print(f'  {f}: {arr.shape} dtype={arr.dtype}')